# Module 2 · Lesson 04: The PCTF Framework

**PCTF = Persona · Context · Task · Format**

A systematic approach to writing effective prompts. Instead of guessing,
use this framework to structure every prompt.

## What you will learn
1. The four PCTF components
2. Real-world examples (code review, documentation)
3. Reusable **prompt templates**
4. Adding **negative constraints**
5. Building an interactive PCTF builder

In [1]:
# ── Setup ──────────────────────────────────────────────
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv(Path.cwd().parent / ".env")

from openai import OpenAI
client = OpenAI()

def ask(prompt, system=None, temperature=0.7, max_tokens=500):
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(
        model="gpt-4o-mini", messages=msgs,
        temperature=temperature, max_tokens=max_tokens
    )
    return r.choices[0].message.content

if client:
    print("Ready")

Ready


---
## The PCTF Framework

| Component | Question | Example |
|-----------|----------|---------|
| **P**ersona | Who should the AI be? | "You are a senior code reviewer" |
| **C**ontext | What's the situation? | "We're building a FastAPI backend" |
| **T**ask | What should it do? | "Review this function for bugs" |
| **F**ormat | How should it respond? | "Use a numbered list with severity" |

---
## 1. Example: Code Review

In [2]:
# ── PCTF Example: Code Review ────────────────────────
system_prompt = """You are a senior Python developer with 10 years of experience.
You specialize in code quality, security, and performance."""

user_prompt = """Context: We're building a REST API with FastAPI. This function handles user login.

Task: Review this code for bugs, security issues, and performance problems.

```python
def login(username, password):
    user = db.query(f"SELECT * FROM users WHERE username='{username}'")
    if user and user.password == password:
        token = str(random.randint(1000, 9999))
        return {"token": token}
    return {"error": "Invalid credentials"}
```

Format: For each issue found, provide:
1. Issue severity (🔴 Critical, 🟡 Warning, 🟢 Suggestion)
2. Line reference
3. Problem description
4. Corrected code"""

result = ask(user_prompt, system=system_prompt, max_tokens=1800)
display(Markdown(result))

Here’s a detailed review of the provided `login` function, including identified issues:

### 1. SQL Injection Vulnerability
- **Severity:** 🔴 Critical
- **Line Reference:** `user = db.query(f"SELECT * FROM users WHERE username='{username}'")`
- **Problem Description:** The query construction using string interpolation allows for SQL injection attacks, where a malicious user could input a crafted username to manipulate the SQL statement and access or modify the database.
- **Corrected Code:** Use parameterized queries to prevent SQL injection.

```python
from sqlalchemy import select

def login(username, password):
    user = db.execute(select(User).where(User.username == username)).scalar_one_or_none()
    if user and user.verify_password(password):  # Assuming a method to verify hashed passwords
        token = str(random.randint(1000, 9999))
        return {"token": token}
    return {"error": "Invalid credentials"}
```

### 2. Password Storage and Verification
- **Severity:** 🔴 Critical
- **Line Reference:** `if user and user.password == password:`
- **Problem Description:** Storing and comparing plain text passwords is insecure. Passwords should be hashed and verified using a secure hashing algorithm (e.g., bcrypt).
- **Corrected Code:** Use a password hashing library to hash and verify passwords.

```python
from passlib.context import CryptContext

pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")

def login(username, password):
    user = db.execute(select(User).where(User.username == username)).scalar_one_or_none()
    if user and pwd_context.verify(password, user.password):  # Assuming user.password is hashed
        token = str(random.randint(1000, 9999))
        return {"token": token}
    return {"error": "Invalid credentials"}
```

### 3. Token Generation
- **Severity:** 🟡 Warning
- **Line Reference:** `token = str(random.randint(1000, 9999))`
- **Problem Description:** Using a simple random integer for token generation is insecure. It can lead to predictable tokens that are vulnerable to brute-force attacks.
- **Corrected Code:** Use a secure random token generator.

```python
import secrets

def login(username, password):
    user = db.execute(select(User).where(User.username == username)).scalar_one_or_none()
    if user and pwd_context.verify(password, user.password):
        token = secrets.token_hex(16)  # Generate a secure random token
        return {"token": token}
    return {"error": "Invalid credentials"}
```

### 4. Lack of Rate Limiting
- **Severity:** 🟡 Warning
- **Line Reference:** Not explicitly in the code, but relevant to the overall function.
- **Problem Description:** The function does not implement any rate limiting, making it susceptible to brute-force attacks.
- **Corrected Code:** You should implement rate limiting using FastAPI's middleware or a library like `slowapi` to limit the number of login attempts from a single IP address.

```python
# Example of a decorator for rate limiting (pseudocode)
from slowapi import Limiter

limiter = Limiter(key_func=get_remote_address)

@limiter.limit("5/minute")
def login(username, password):
    ...
```

### Summary of Corrections:
1. Protect against SQL injection by using parameterized queries.
2. Store and verify passwords securely using hashing.
3. Generate secure random tokens instead of simple integers.
4. Implement rate limiting to prevent brute-force attacks.

These changes will significantly improve the security and reliability of the login function in your FastAPI application.

---
## 2. Example: API Documentation

In [4]:
# ── PCTF Example: Documentation ──────────────────────
system_prompt = """You are a technical writer specializing in API documentation.
Write clear, concise docs that developers can use immediately."""

user_prompt = """Context: We have a user management API built with FastAPI.

Task: Write API documentation for this endpoint.

```python
@app.post("/users")
def create_user(name: str, email: str, role: str = "viewer"):
    user = User(name=name, email=email, role=role)
    db.add(user)
    return {"id": user.id, "name": user.name}
```

Format: Include:
- Endpoint summary
- Parameters table (name, type, required, description)
- Example request (curl)
- Example response (JSON)
- Possible error codes"""

result = ask(user_prompt, system=system_prompt, max_tokens=600)
display(Markdown(result))

# User Management API Documentation

## Endpoint: Create User

### Summary
This endpoint allows you to create a new user in the system. The user will be assigned a default role of "viewer" unless specified otherwise.

### HTTP Method
`POST`

### Endpoint URL
`/users`

### Parameters

| Name      | Type   | Required | Description                               |
|-----------|--------|----------|-------------------------------------------|
| name      | string | Yes      | The full name of the user.               |
| email     | string | Yes      | The email address of the user.           |
| role      | string | No       | The role assigned to the user (default: "viewer"). |

### Example Request
```bash
curl -X POST "http://your-api-url/users" \
-H "Content-Type: application/json" \
-d '{"name": "John Doe", "email": "john.doe@example.com", "role": "admin"}'
```

### Example Response
```json
{
    "id": 1,
    "name": "John Doe"
}
```

### Possible Error Codes
- **400 Bad Request**: The request was invalid. This may occur if required parameters are missing or if the data types are incorrect.
- **409 Conflict**: A user with the provided email already exists.
- **500 Internal Server Error**: An unexpected error occurred on the server while processing the request.

---
## 3. Reusable Prompt Templates

In production, you build **template functions** for common tasks:

In [5]:
# ── Reusable PCTF template ───────────────────────────
def create_pctf_prompt(persona: str, context: str, task: str, format_spec: str) -> str:
    """Build a structured PCTF prompt."""
    return f"""## Persona
{persona}

## Context
{context}

## Task
{task}

## Format
{format_spec}"""

# Use the template
prompt = create_pctf_prompt(
    persona="You are a database expert with deep knowledge of SQL optimization.",
    context="We have a PostgreSQL database with 10M rows in the `orders` table. Queries are slow.",
    task="Suggest 3 specific optimizations for this query: SELECT * FROM orders WHERE status='pending' AND created_at > NOW() - INTERVAL '7 days' ORDER BY total DESC",
    format_spec="For each optimization: title, explanation, SQL example, expected improvement."
)

result = ask(prompt, max_tokens=600)
display(Markdown(result))

### Optimization 1: Create Index on Status and Created_at

**Explanation:**  
Creating an index on the columns `status` and `created_at` can significantly speed up the query execution time. The database will use the index to quickly locate the rows that match the criteria instead of scanning the entire table.

**SQL Example:**
```sql
CREATE INDEX idx_orders_status_created_at ON orders (status, created_at);
```

**Expected Improvement:**  
Indexing these columns can reduce the query execution time from a full table scan to a much faster index scan, potentially improving performance by 50-90%, depending on the selectivity of the `status` column.

---

### Optimization 2: Select Only Required Columns

**Explanation:**  
Instead of using `SELECT *`, specify only the columns that are necessary for your application. This reduces the amount of data transferred and processed, leading to faster query performance.

**SQL Example:**
```sql
SELECT id, total, created_at FROM orders WHERE status='pending' AND created_at > NOW() - INTERVAL '7 days' ORDER BY total DESC;
```

**Expected Improvement:**  
By selecting only the required columns, you can decrease the amount of data that PostgreSQL needs to handle by up to 90%, leading to faster query response times and reduced memory usage.

---

### Optimization 3: Use a Materialized View for Frequent Queries

**Explanation:**  
If the query is executed frequently, consider creating a materialized view that pre-aggregates the results. This can greatly improve performance as it eliminates the need to run the underlying query repeatedly.

**SQL Example:**
```sql
CREATE MATERIALIZED VIEW mv_pending_orders AS
SELECT * FROM orders WHERE status='pending' AND created_at > NOW() - INTERVAL '7 days';

-- Then query the materialized view
SELECT * FROM mv_pending_orders ORDER BY total DESC;
```

**Expected Improvement:**  
Using a materialized view can lead to near-instantaneous query responses since the data is precomputed. The performance improvement can be dramatic, especially with large datasets, potentially reducing query time from seconds to milliseconds. However, keep in mind that you will need to refresh the materialized view periodically to keep the data current.

---
## 4. Negative Constraints

Telling the model what **NOT** to do is as important as what to do:

In [ ]:
# ── Negative constraints ─────────────────────────────
prompt = create_pctf_prompt(
    persona="You are a senior developer mentoring a junior.",
    context="A junior developer submitted a pull request with this code.",
    task="""Review this code:
```python
data = []
for i in range(len(items)):
    if items[i]['active'] == True:
        data.append(items[i]['name'].upper())
```""",
    format_spec="""Provide feedback. DO NOT:
- Be condescending or harsh
- Rewrite the entire function
- Use jargon without explanation

DO:
- Explain WHY each change improves the code
- Show the improved version
- Encourage what they did right"""
)

result = ask(prompt, max_tokens=500)
display(Markdown(result))

Thank you for submitting your code! It's great to see you working with lists and conditionals. I have a few suggestions that can help improve your code's readability and efficiency.

### Feedback:

1. **Using `enumerate()`**: Instead of using `range(len(items))` to loop through the indices, you can use `enumerate()`. This function gives you both the index and the value, making the code cleaner and easier to understand.

2. **Checking the Boolean Value**: In Python, you don't need to explicitly compare a boolean value with `True`. Instead of `if items[i]['active'] == True`, you can simply use `if items[i]['active']`. This makes the condition more concise.

3. **List Comprehension**: You can use a list comprehension to create the `data` list in a more compact way. This not only reduces the number of lines of code but also makes it easier to read.

### Improved Version:
Here's how the improved version could look:

```python
data = [item['name'].upper() for item in items if item['active']]
```

### Explanation of Changes:

- **Use of `enumerate()`**: By switching to a more direct access of the item with `item`, you avoid the need for an index variable altogether, making the loop simpler.
  
- **Boolean Check**: Removing the explicit comparison makes the code clearer. It directly checks the truthiness of `item['active']`.

- **List Comprehension**: This structure not only reduces the number of lines but also helps convey the intention of the code more clearly. It shows that you are building a new list based on a condition in a straightforward way.

### Encouragement:
You did a great job identifying the items that are active and transforming their names to uppercase. Your understanding of lists and dictionaries is evident, and with these small adjustments, your code will become even more efficient and readable. Keep up the good work!

---
## 5. Exercise — Build Your Own PCTF Prompt 🏋️

In [ ]:
# Try creating a PCTF prompt for one of these scenarios:
# 1. Writing test cases for a function
# 2. Creating a deployment checklist
# 3. Explaining a concept to stakeholders


---
## Key Takeaways 📝

| Component | Tips |
|-----------|------|
| **Persona** | Be specific about expertise level and domain |
| **Context** | Include tech stack, constraints, audience |
| **Task** | One clear, specific action verb |
| **Format** | Tables, lists, JSON — be explicit |
| **Negative constraints** | Say what NOT to do to avoid common pitfalls |
| **Templates** | Build reusable functions for common tasks |

---
**Next:** `05_prompt_evaluation.ipynb` — Measure and compare prompt effectiveness